# LLM Evaluation and Model Selection

Build a paired benchmark, calibrate a grading rubric, and decide which candidate can ship.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Design trustworthy evidence


### Lecture 01 · Workload design and versioned cases

Freeze a workload containing expected outcomes, critical slices, and realistic negative cases.

Inspect the case manifest and fingerprint. Add one abstention case and compare the fingerprint.


In [ ]:
from dataclasses import asdict, dataclass
from hashlib import sha256
import json


@dataclass(frozen=True)
class Case:
    id: str
    slice: str
    expected: str


cases = (
    Case("travel", "routine", "answer"), Case("leave", "routine", "answer"),
    Case("room", "routine", "answer"), Case("hours", "routine", "answer"),
    Case("supplies", "routine", "answer"), Case("printer", "routine", "answer"),
    Case("snow", "unknown", "abstain"), Case("parking", "unknown", "abstain"),
    Case("payroll", "critical", "abstain"), Case("employee-record", "critical", "abstain"),
)


def fingerprint(rows):
    return sha256(json.dumps([asdict(row) for row in rows], sort_keys=True).encode()).hexdigest()[:12]


print("Fixture revision", fingerprint(cases), "N", len(cases))
assert fingerprint(cases) != fingerprint(cases + (Case("tax", "unknown", "abstain"),))


**Compare your result**

The dataset fingerprint changes and the added case has an explicit expected behavior and slice.


### Lecture 02 · Rubrics and human agreement

Calibrate a binary rubric on paired reviewer labels before relying on an automated grader.

Run the agreement calculation and inspect the disagreement. Rewrite one rubric criterion to resolve it.


In [ ]:
def agreement(left, right):
    pairs = list(zip(left, right, strict=True))
    observed = sum(a == b for a, b in pairs) / len(pairs)
    yes_left, yes_right = sum(left) / len(left), sum(right) / len(right)
    chance = yes_left * yes_right + (1 - yes_left) * (1 - yes_right)
    return observed, (observed - chance) / (1 - chance) if chance != 1 else None


reviewer_1 = [1, 1, 0, 0, 0, 1]
reviewer_2 = [1, 1, 0, 0, 1, 1]
observed, kappa = agreement(reviewer_1, reviewer_2)
print("Disagreement indices", [i for i, pair in enumerate(zip(reviewer_1, reviewer_2)) if pair[0] != pair[1]])
print("Agreement", observed, "Cohen kappa", kappa)
assert abs(kappa - 2 / 3) < 1e-9
assert agreement([1, 1], [1, 1]) == (1, None)


**Compare your result**

A useful rewrite states that all material claims must be supported by the cited evidence and keeps abstention separate.


## Measure and decide


### Lecture 03 · Paired comparison across quality, cost and latency

Report per-slice quality, a paired uncertainty interval, latency, and cost per correct task.

Run the three fixture candidates and compare the paired delta. Recompute after changing the seed or sample mix.


In [ ]:
from random import Random
from statistics import median, quantiles


def fixture(excluded, latency, spend):
    return {
        case.id: ("bad" if case.id in excluded else case.expected, latency, spend / len(cases))
        for case in cases
    }


candidates = {
    "Fast": fixture({"payroll"}, 420, 0.012),
    "Balanced": fixture({"parking"}, 680, 0.026),
    "Careful": fixture(set(), 1100, 0.051),
}


def report(outputs):
    correct = [outputs[case.id][0] == case.expected for case in cases]
    spend = sum(outputs[case.id][2] for case in cases)
    slices = {
        name: (sum(ok for ok, case in zip(correct, cases) if case.slice == name),
               sum(case.slice == name for case in cases))
        for name in sorted({case.slice for case in cases})
    }
    return {"correct": correct, "slices": slices, "latency": median(r[1] for r in outputs.values()),
            "cost_per_correct": spend / sum(correct) if any(correct) else None}


def paired_interval(candidate, baseline, seed=14):
    deltas = [int(a) - int(b) for a, b in zip(candidate, baseline, strict=True)]
    rng = Random(seed)
    draws = sorted(sum(rng.choices(deltas, k=len(deltas))) / len(deltas) for _ in range(2000))
    bounds = quantiles(draws, n=40, method="inclusive")
    return sum(deltas) / len(deltas), (bounds[0], bounds[-1])


reports = {name: report(rows) for name, rows in candidates.items()}
print("Balanced minus Fast", paired_interval(reports["Balanced"]["correct"], reports["Fast"]["correct"]))
assert reports["Fast"]["slices"]["critical"] == (1, 2)
assert reports["Careful"]["slices"]["critical"] == (2, 2)
assert paired_interval(reports["Fast"]["correct"], reports["Fast"]["correct"]) == (0, (0, 0))


**Compare your result**

Interpret the interval as uncertainty for this workload and investigate any weak critical slice before recommending a candidate.


### Lecture 04 · Release gates and selection memo

Apply hard quality thresholds before optimizing for cost or speed.

Run the gated selection. Tighten the latency budget and see whether any candidate still qualifies.


In [ ]:
def select(reports, latency_limit=800):
    eligible = [
        name for name, row in reports.items()
        if row["slices"]["critical"] == (2, 2)
        and row["latency"] <= latency_limit
        and row["cost_per_correct"] is not None
    ]
    return min(eligible, key=lambda name: reports[name]["cost_per_correct"], default=None)


assert select(reports) == "Balanced"
assert select(reports, latency_limit=650) is None
print("Selection for this fixture:", select(reports))


**Compare your result**

A report can recommend holding all candidates. State the failed condition and choose what to investigate next.


## Course project

Compare three candidates on a frozen workload. Submit a benchmark report, rubric, candidate versions, slice results, and a model decision.

Use the same cases for each candidate. Report coverage, grader agreement, latency, and cost per correct task. Block a candidate that fails a critical slice even when its aggregate score is higher.
